<div class="blog-language-switch" role="group" aria-label="Article language">
<span aria-current="page">English</span>
<a href="/ipynb/zh-CN/Computer-Science/Operating-Systems/09-file-systems-persistence-and-crash-consistency.html" lang="zh-CN" hreflang="zh-CN">中文</a>
</div>

[Back to Operating Systems guideline](Operating-Systems.html)


## **File Systems, Persistence, and Crash Consistency**

The previous chapter followed a cache-missing `read()` until the I/O stack submitted a device request and delivered its completion. One box was deliberately left abstract: **how does a pathname and byte offset identify the persistent blocks that should be read or changed?** This chapter opens that box.

A file system presents named objects that can outlive the process that created them. It lets several processes share those objects, maps logical byte ranges onto storage, delays and batches writes for performance, and reconstructs a valid state after interruption. These responsibilities interact. Creating one file may change a directory block, a free-inode map, an inode, a free-block map, file data, and a recovery log. The hardware normally persists these blocks separately, so a power failure can interrupt the transition between them.

Persistence should therefore not be confused with "calling `write()`" or "putting data on a disk." It is a contract involving:

- the **namespace**, which determines which names reach which objects;
- the **object model**, which defines files, directories, links and metadata;
- the **mapping and allocation structures**, which identify owned storage;
- the **cache and ordering protocol**, which determines when changes leave volatile memory; and
- the **recovery design**, which determines which state is accepted after failure.

A useful mental model is:

> **A file system transforms named logical updates into an ordered persistent-state transition, while preserving enough evidence to recover a valid state after any failure covered by its model.**

This chapter focuses on local file-system mechanisms. Distributed file systems add server failure, network partition, client caching, leases and consensus questions. Protection details such as ACLs, capabilities and sandboxing remain in the next chapter; here, permissions appear only where they affect ordinary file API and path-walk semantics.

### **What Persistence Must Provide**

**Persistent** does not mean immortal. It means that an acknowledged state survives a stated set of events, such as process termination, kernel crash, reboot or power loss, provided the storage device behaves according to its contract. Device loss, media corruption, malicious modification, operator error and destruction of every replica require additional integrity, redundancy and backup mechanisms.

![Persistent state crosses application, namespace, filesystem, cache, device, and recovery contracts.](assets/fs-persistence-contract-layers.svg){fig-alt="Layers connect application update intent to VFS naming, filesystem structures, cached write transport, durable storage, and recovery, with errors and recovered state flowing upward." width="96%"}

*Figure: original explanatory diagram based on [OSTEP file-system implementation](https://pages.cs.wisc.edu/~remzi/OSTEP/file-implementation.pdf), Linux [VFS documentation](https://docs.kernel.org/filesystems/vfs.html), and Linux [`fsync(2)`](https://man7.org/linux/man-pages/man2/fsync.2.html).*

Four properties are especially easy to blur:

| Property | Question it answers | Example failure |
|---|---|---|
| **Consistency** | Do related structures satisfy their invariants? | A block is marked free while an inode points to it |
| **Durability** | Which acknowledged updates survive the stated crash/power model? | `write()` returned, but unsynchronized dirty pages disappear after power loss |
| **Integrity** | Can accidental or malicious corruption be detected or corrected? | A valid-looking block contains silently changed bytes |
| **Availability** | Can clients continue to access data despite faults? | The only device fails even though its pre-failure structures were consistent |

Journaling primarily targets consistency and bounded recovery time. `fsync()` requests a durability boundary for selected state. Checksums improve corruption detection. Replication, RAID and backups address different availability and loss scenarios. One mechanism does not automatically provide the other properties.

The failure model must also say what storage guarantees. Relevant questions include:

- Can the device reorder independent writes?
- Can a completed host command remain only in volatile device cache?
- What does a cache-flush or force-unit-access operation guarantee?
- What is the atomic write unit, and can a larger block tear?
- Are stale, misplaced or silently corrupted writes possible?
- Is a remote server acknowledgement equivalent to stable storage?

File systems use ordering barriers, flushes, checksums, replicated metadata, sequence numbers and recovery records to strengthen weaker primitives. Applications must then use the file-system API correctly; the strongest journal cannot infer that an application intended several separately opened files to form one transaction.

#### **Naming, Sharing, Durability, and Recovery**

**Naming** converts a human-meaningful pathname such as `/home/ana/report.txt` into a sequence of object lookups. The hierarchy provides organization and allows a directory subtree to be moved or mounted without changing each file's internal identity. Names are mutable references: a rename can make one name refer to a different object while an already-open descriptor continues to refer to the old object.

**Sharing** lets several names, descriptors, processes and mappings refer to one file object. The file system coordinates link counts, open references, cached pages, byte-range state and concurrent updates. Sharing makes lifetime reference-based rather than name-based.

**Durability** establishes a point after which specified data and metadata can be recovered despite a covered failure. Ordinary buffered writes prioritize throughput and latency by returning before storage persistence. Explicit synchronization, synchronous flags or higher-level transactional protocols request stronger boundaries when needed.

**Recovery** chooses a valid post-crash state. A checker can reconstruct global relationships, a journal can replay committed recent changes, and a copy-on-write tree can choose an old or new durable root. Recovery promises structure, not necessarily the application's latest intended business state. If the application acknowledged a payment before synchronizing its record, a structurally perfect old file system can still represent the wrong business outcome.

| Event | File-system responsibility | Application responsibility |
|---|---|---|
| Process exits normally | Keep persistent objects independent of process lifetime | Close/check errors; synchronize state whose loss is unacceptable |
| Process crashes | Release volatile handles/locks while retaining named objects | Use atomic replacement or application log so partial logical updates are recognizable |
| Kernel/power crash | Recover a structurally valid state covered by the filesystem protocol | Place synchronization boundaries before external acknowledgement |
| Media corruption | Detect if checksums/redundancy exist; report or repair if possible | Maintain independent backups and validate critical content |
| Operator deletes/overwrites data | Apply namespace operation as authorized | Use versioning, snapshots with retention, and offline/independent backups |

### **The File and Directory API**

The file API separates operations on **names** from operations on **open objects**:

| Operation family | Representative calls | Object affected |
|---|---|---|
| Resolve/open | `open`, `openat`, `openat2`, `stat`, `fstatat` | Convert pathname/location into metadata or an open handle |
| Byte I/O | `read`, `write`, `pread`, `pwrite`, `lseek`, `mmap` | File contents and possibly the open offset |
| Namespace | `mkdir`, `link`, `symlink`, `unlink`, `rmdir`, `rename` | Directory entries and object link counts |
| Metadata | `chmod`, `chown`, `utimensat`, extended-attribute calls | Inode/object attributes |
| Persistence | `fsync`, `fdatasync`, synchronous open/write flags | Ordering and durable completion boundary |
| Space control | `ftruncate`, `fallocate`, hole punching | Logical size and physical allocation policy |

The `*at()` interfaces accept a directory descriptor plus a relative path. This is not just syntactic convenience. A stable directory handle avoids dependence on a process-wide current working directory and reduces races where an attacker or concurrent thread replaces an earlier path component between separate checks and use. Linux `openat2()` can impose additional resolution constraints, but the general design lesson is portable: operate relative to an already-resolved directory object when a sequence must stay anchored.

System-call atomicity is operation-specific. `O_CREAT | O_EXCL` combines existence testing and creation. `O_APPEND` combines finding the end offset with each write on a suitable local file-system implementation. A same-filesystem `rename()` atomically changes namespace visibility so observers do not see the destination missing between removal and replacement. None of these statements alone says that the operation is already durable after sudden power loss.

#### **Files, Directories, Links, and Metadata**

A regular file is a byte sequence plus metadata. A directory is a special file-system object whose data maps component names to object identities. A **hard link** adds another directory entry for the same inode/object. A **symbolic link** is a distinct inode whose contents are another pathname to resolve.

![Hard links share an inode, a symbolic link stores a pathname, and an open handle can outlive every name.](assets/fs-links-object-lifetime.svg){fig-alt="Directory entries report and backup both point to inode 42, a symlink stores a textual path, and after both hard links are unlinked an open file description retains the inode until the last descriptor closes." width="96%"}

*Figure: original explanatory diagram based on Linux [`link(2)`](https://man7.org/linux/man-pages/man2/link.2.html), [`unlink(2)`](https://man7.org/linux/man-pages/man2/unlink.2.html), and the ext4 [inode format](https://docs.kernel.org/filesystems/ext4/inodes.html).*

Hard and symbolic links behave differently:

| Property | Hard link | Symbolic link |
|---|---|---|
| Stored relationship | Directory entry directly names the same inode/object | Separate object stores path text |
| Cross-filesystem target | Normally impossible | Possible because the text is resolved later |
| Target removal | Other hard-link names still access the object | Link can become dangling |
| Identity reported by `stat` | Same inode and device as other hard links | `lstat` reports the symlink; `stat` follows target |
| Directory hard links | Normally restricted to preserve hierarchy/invariants | Can point to a directory path |
| Rename of target | Other hard links are unaffected | Relative/absolute text may resolve differently or fail |

`unlink()` removes one **name**. If it was the last hard link but an open file description still refers to the inode, the file remains usable and its storage is reclaimed only after the final reference closes. This property supports safe temporary files and software upgrades: a running process can continue executing or reading an old unlinked object while the namespace points new opens to a replacement.

Metadata commonly exposed by `stat()` includes:

- object type and mode bits;
- owner UID and group GID;
- inode number within a containing filesystem/device identity;
- logical byte size and allocated block count;
- hard-link count;
- modification time (`mtime`), metadata-change time (`ctime`) and access time (`atime`); and
- preferred I/O block size and device information.

On Unix, `ctime` is **change time**, not creation time. Some filesystems expose a birth/creation time through newer interfaces, but code should not infer it from `ctime`. Logical size can greatly exceed allocated blocks for a sparse file, and allocated-block units reported by tools may differ from the filesystem block size.

Directory permissions influence path operations: search/execute permission is needed to traverse a directory; directory write permission governs creating or removing names. Reading a file's bytes and removing its name are therefore different authorities. The next chapter develops this protection model in detail.

#### **File Descriptors and Open File Tables**

`open()` creates an **open file description** and installs a reference to it in the calling process's descriptor table. These layers hold different state:

- the **descriptor entry** has a small integer number and descriptor flags such as close-on-exec;
- the **open file description** has the current file offset and status flags such as append/nonblocking mode; and
- the **inode/file object** has persistent identity, metadata, mapping and cached content state.

![dup and fork can share one open description and offset, while separate open calls have independent offsets.](assets/fd-open-file-inode.svg){fig-alt="Two process descriptor tables point through shared or separate open file descriptions to the same inode; dup and fork share one offset while a distinct open creates another offset." width="96%"}

*Figure: original explanatory diagram based on Linux [`open(2)`](https://man7.org/linux/man-pages/man2/open.2.html) and the kernel [VFS file-object description](https://docs.kernel.org/filesystems/vfs.html).*

`dup()` creates another descriptor for the same open file description. `fork()` normally copies descriptor-table references, so parent and child descriptors still share open offsets and status flags. A second independent `open()` normally creates a separate open description and offset even when it reaches the same inode. `pread()` and `pwrite()` avoid mutating a shared open offset by taking an explicit position.

The following program makes those relationships observable. It creates a file, duplicates one descriptor, independently opens the same pathname, removes the pathname, and continues reading through all open references.

<details>
<summary><strong>C example: shared offsets and an unlinked-but-open file</strong></summary>

```c
#define _POSIX_C_SOURCE 200809L
#include <errno.h>
#include <fcntl.h>
#include <stdint.h>
#include <stdio.h>
#include <stdlib.h>
#include <sys/stat.h>
#include <unistd.h>

static void die(const char *what) {
    perror(what);
    exit(EXIT_FAILURE);
}

static char read_one(int fd) {
    char value;
    for (;;) {
        ssize_t n = read(fd, &value, 1);
        if (n == 1) return value;
        if (n < 0 && errno == EINTR) continue;
        if (n == 0) {
            fprintf(stderr, "unexpected EOF\n");
            exit(EXIT_FAILURE);
        }
        die("read");
    }
}

int main(void) {
    char path[] = "/tmp/open-state-XXXXXX";
    int original = mkstemp(path);       // Creates and opens one unique file.
    if (original == -1) die("mkstemp");

    const char contents[] = "ABCDE";
    if (write(original, contents, sizeof contents - 1) !=
        (ssize_t)(sizeof contents - 1)) {
        die("write");
    }
    if (lseek(original, 0, SEEK_SET) == -1) die("lseek");

    int duplicate = dup(original);      // Shares original's open description/offset.
    if (duplicate == -1) die("dup");

    int independent = open(path, O_RDONLY | O_CLOEXEC); // New open description.
    if (independent == -1) die("open independent");

    if (unlink(path) == -1) die("unlink"); // Remove the only directory name.

    struct stat status;
    if (fstat(original, &status) == -1) die("fstat");
    printf("link count after unlink: %ju\n", (uintmax_t)status.st_nlink);

    // original reads 'A' and advances the shared offset to 1.
    printf("original:    %c\n", read_one(original));
    // duplicate starts at that shared offset and therefore reads 'B'.
    printf("duplicate:   %c\n", read_one(duplicate));
    // independent has its own offset, initially zero, and reads 'A'.
    printf("independent: %c\n", read_one(independent));

    close(independent);
    close(duplicate);
    close(original);                     // Last reference: storage is reclaimable.
    return EXIT_SUCCESS;
}
```

Compile and run on a POSIX-like system:

```bash
cc -std=c11 -Wall -Wextra -O2 open_state.c -o open_state
./open_state
```

Expected relationship (temporary pathname differs each run):

```text
link count after unlink: 0
original:    A
duplicate:   B
independent: A
```

</details>

Production code must check `close()` when earlier writes matter because delayed writeback errors can be reported late, although `close()` is not a substitute for `fsync()` when crash durability is required. `O_CLOEXEC` should normally be requested atomically during descriptor creation so a concurrent `fork()`/`exec()` cannot leak a descriptor between `open()` and a later `fcntl()`.

### **Inodes and Directory Structures**

An **inode** is the file-system record for an object. In a classic Unix design it stores type, size, ownership, mode, timestamps, link count, flags, and a map from logical file blocks to storage. It normally does **not** store the filename. A directory entry stores a name and inode number/object identity; several directory entries can therefore point to one inode.

Path lookup alternates between these two structures. Resolving `/home/ana/report.txt` conceptually performs:

1. begin at the root dentry/inode of the caller's mount namespace;
2. read or consult cached root-directory entries to find `home`;
3. load/consult that inode and verify that it is a searchable directory;
4. repeat for `ana`;
5. find `report.txt` in the final directory;
6. load the final inode and apply the operation-specific checks; and
7. for data I/O, use the inode's mapping to translate logical offsets.

![Path lookup alternates through directory name mappings and inodes until the final inode maps file offsets.](assets/pathname-lookup-animated.svg){fig-alt="An animated lookup token follows /home/ana/report.txt from root inode to root directory, home inode and directory, ana inode and directory, final file inode, and logical data extents." width="97%"}

*Figure: original explanatory animation informed by [UW CS537 directory notes](https://pages.cs.wisc.edu/~bart/537/lecturenotes/s25.html), Linux [pathname lookup](https://docs.kernel.org/filesystems/path-lookup.html), and Linux [VFS documentation](https://docs.kernel.org/filesystems/vfs.html).*

Real path lookup includes `.` and `..`, symbolic links, mount points, permission checks, concurrent rename/unmount, negative cache entries, automounts and process-specific roots. Linux first attempts a highly cache-oriented RCU walk and falls back to a reference-taking walk for cases that need blocking or stronger stabilization. The implementation is complex because lookup must be fast while the namespace changes concurrently.

Directories can be represented as:

| Representation | Lookup | Update | Appropriate scale |
|---|---|---|---|
| Linear directory entries | $O(n)$ name comparisons | Simple insertion/removal with free-record management | Small directories and simple teaching formats |
| Hash table / hashed tree | Average near $O(1)$ or bounded tree lookup | Must handle collisions, splits and stable enumeration semantics | Large general-purpose directories such as ext4 indexed directories |
| B-tree/B+tree | $O(\log_B n)$ node accesses with high fanout $B$ | Split/merge or copy-on-write updates | Large directories and tree-centric filesystems |

Caching often dominates the on-disk asymptotic difference. A positive dentry cache remembers a successful name-to-inode relationship; a **negative dentry** remembers that a name did not exist, accelerating repeated misses and later creation paths. Cache entries are performance state, not the persistent directory itself, and must be invalidated or revalidated around namespace changes.

For a filesystem block size $b$ and byte offset $x$, the logical file block and intra-block offset are

$$
\operatorname{lbn} = \left\lfloor \frac{x}{b} \right\rfloor,
\qquad
\operatorname{within} = x \bmod b.
$$

The inode's mapping translates `lbn` into a physical/logical storage block. Older Unix designs use direct and multi-level indirect pointers. Modern local file systems often use **extents**, where one record maps a contiguous run `(logical_start, length)` to `(physical_start, length)`.

![An inode points to an extent index that maps logical ranges, including unallocated sparse holes.](assets/inode-extent-mapping.svg){fig-alt="An inode stores object metadata and the root of an extent tree; extent leaves map logical block ranges to physical ranges while sparse holes have no allocated blocks and read as zeros." width="96%"}

*Figure: original explanatory diagram based on the ext4 [inode and extent format](https://docs.kernel.org/filesystems/ext4/inodes.html) and [OSTEP file-system implementation](https://pages.cs.wisc.edu/~remzi/OSTEP/file-implementation.pdf).*

Extents compress metadata and improve sequential locality when files occupy long runs. Fragmentation creates more extent records and potentially deeper trees. A **sparse hole** is a logical range with no data blocks allocated; reads synthesize zeros, and a later write allocates blocks only for the touched region. Consequently, file size is a logical boundary, not a direct measure of consumed storage.

### **On-Disk File-System Layout**

Formatting a volume creates enough persistent metadata to interpret every later block. A teaching Unix-like layout often includes a superblock, allocation maps, inode records, directory/data blocks and optional recovery log. General-purpose file systems distribute these structures into allocation groups or block groups to improve locality, parallelism and damage containment.

![A representative ext4 volume contains block groups with metadata, allocation maps, inode tables, data blocks, and a journal.](assets/ext4-on-disk-layout.svg){fig-alt="An ext4 teaching layout divides a volume into block groups; each representative group contains superblock/group metadata, block and inode bitmaps, inode tables, and data blocks, while a reserved journal contains transaction records." width="97%"}

*Figure: original explanatory diagram based on the official ext4 [high-level design](https://docs.kernel.org/filesystems/ext4/overview.html), [block-group layout](https://docs.kernel.org/filesystems/ext4/blockgroup.html), and Oracle Linux's [ext4 layout explanation](https://blogs.oracle.com/linux/understanding-ext4-disk-layout-part-1).*

Block grouping improves locality: placing a file's inode, directory entry and data in the same or nearby group reduces seek cost on disks and improves cache/queue behavior more generally. It also avoids one giant centralized inode table or bitmap becoming a hot location. Large files may deliberately spread across groups so one file does not consume all space local to its directory.

This ext-family picture is representative, not universal. XFS uses allocation groups and B+trees. Btrfs consists largely of copy-on-write trees. FAT centralizes an allocation table. Log-structured systems treat the log and checkpoints as the primary layout. Network and memory-backed filesystems may not have a local block layout at all.

#### **Superblocks, Inode Tables, Data Blocks, and Logs**

Each on-disk structure answers a different reconstruction question:

| Structure | Core contents | Key invariant |
|---|---|---|
| Superblock | Format version/features, block size, volume identity, geometry, root/recovery pointers | Kernel must know how to interpret all other blocks |
| Group/allocation descriptor | Locations and summary counts for a region | Summary must agree with underlying maps or be repairable |
| Inode bitmap/table | Which inode slots are allocated and each object's metadata/map | Allocated inode identity is unique and references valid owned blocks |
| Block/free-space map | Which storage ranges are free or owned | No live block is simultaneously free or multiply owned without explicit sharing |
| Directory block/index | Component names mapped to inode identities | Entry targets valid objects; link counts reflect supported naming relationships |
| Data block | User bytes or directory/index payload | Reachability and ownership agree with metadata |
| Journal/log | Recent transaction records, sequence/checksum and commit evidence | Recovery can distinguish complete committed records from incomplete tail data |

For ext4, inode numbers locate fixed-size records within per-group inode tables. If `ino` is a nonzero inode number, $N_g$ is the number of inodes per group, and $S_i$ is the inode-record size, then

$$
\operatorname{group} = \left\lfloor\frac{ino - 1}{N_g}\right\rfloor,
\qquad
\operatorname{index} = (ino - 1) \bmod N_g,
\qquad
\operatorname{byteOffset} = \operatorname{index} S_i.
$$

This arithmetic finds the record; the inode's extent/pointer tree then finds file contents. Directory indexes find inode numbers, not physical data blocks directly.

Checksums help detect torn, stale or corrupted metadata. They do not by themselves prove global consistency: a perfectly checksummed bitmap can still disagree with a perfectly checksummed inode. Sequence numbers, ownership fields, reverse maps and recovery protocols add cross-structure evidence. Repair needs to decide which valid-looking copy or relationship is authoritative.

### **File Allocation and Free-Space Management**

Allocation connects a file's logical address space to finite storage. The allocator must answer two questions:

1. **Which free units should this file receive?**
2. **How will the file's mapping remember those units efficiently?**

The choices affect sequential throughput, random-access cost, metadata size, future growth, fragmentation, crash dependencies and cleaning work.

#### **Contiguous, Linked, Indexed, and Extent-Based Allocation**

![Contiguous, linked, indexed, and extent mappings make different growth, lookup, and fragmentation trade-offs.](assets/file-allocation-strategies.svg){fig-alt="Four panels compare contiguous allocation, linked blocks, an index block, and extent range records; a lower panel compares free-space bitmaps and free-extent trees." width="98%"}

*Figure: original explanatory diagram based on [OSTEP file-system implementation](https://pages.cs.wisc.edu/~remzi/OSTEP/file-implementation.pdf), [OSTEP Fast File System](https://pages.cs.wisc.edu/~remzi/OSTEP/file-ffs.pdf), and the ext4 [extent format](https://docs.kernel.org/filesystems/ext4/inodes.html).*

| Scheme | Mapping metadata | Access to logical block $i$ | Growth behavior | Main weakness |
|---|---|---|---|---|
| Contiguous | Start block + length | $O(1)$ arithmetic | Difficult if adjacent space is occupied | External fragmentation or relocation/over-allocation |
| Linked | Each block points to next, or a central FAT records links | $O(i)$ through chain without auxiliary cache/index | Easy to add any free block | Poor random access, pointer overhead, chain corruption |
| Indexed | Array/tree of block pointers | $O(1)$ for a resident direct index; $O(h)$ I/O for tree height $h$ | Add pointers and index levels | Index-block overhead, especially for tiny files |
| Extent-based | Tree of `(logical start, physical start, length)` | Typically $O(\log_B E)$ for $E$ extents and fanout $B$ | Efficient while long free runs exist | Fragmentation creates more extents and tree updates |

Contiguous allocation is ideal for a fixed-size, read-mostly image because one record maps the whole file. It is poor for unpredictable growth. Linked allocation grows easily but turns random access into a chain traversal. Indexed allocation gives direct lookup but may spend a full index block on a small file. Extents combine indexing with run-length compression and are widely used in modern general-purpose file systems.

**Internal fragmentation** is unused space inside an allocated unit. A 4097-byte file with 4096-byte blocks needs two blocks and leaves almost one full block unused unless tail packing or inline data is supported. For a file of size $F$ and allocation unit $b$, simple full-block allocation consumes

$$
A = \left\lceil \frac{F}{b} \right\rceil b,
\qquad
W_{internal} = A - F.
$$

**External fragmentation** means free capacity exists but is split into runs unsuitable for a desired allocation. Extent trees describe a fragmented file correctly, but more extents enlarge metadata, reduce sequential opportunities and increase update work. Fragmentation is therefore a performance and metadata problem even when no bytes are logically lost.

Free-space representations include:

- **bitmaps**, with one bit per block/cluster/inode, which are compact, easy to count and efficient for word-level searches for runs;
- **free lists**, simple but weak for finding a run of a desired size;
- **free-extent trees**, compact for long runs and capable of best-fit/nearby searches, but requiring split/merge updates; and
- **space summaries**, maintained per group/region to avoid scanning the whole volume.

An allocator is also a locality policy. It may place a new file near its parent directory, place related inodes together, reserve room for expected growth, spread very large files, choose a CPU-local allocation group, or separate hot and cold data. Disk-era cylinder groups reduced seeks; the same grouping still improves metadata locality, queue parallelism and failure containment on newer devices.

**Delayed allocation** postpones choosing physical blocks until writeback, when the filesystem can see a larger dirty range and allocate a better extent. It improves layout and avoids allocating data that is overwritten or deleted while still dirty. The trade-off is that a successful buffered `write()` may later encounter `ENOSPC` or quota errors during writeback or `fsync()`. **Preallocation** (`fallocate()` on supporting systems) reserves space earlier so future writes are less likely to fragment or fail for capacity.

A sparse file demonstrates the distinction between logical and allocated space:

```bash
# Logical size becomes 1 GiB, but no data blocks are needed for the hole itself.
truncate -s 1G sparse.img

# ls/stat report logical size; du reports allocated storage.
ls -lh sparse.img
du -h sparse.img
stat sparse.img
```

Writing one byte near the end normally allocates only the touched block plus metadata. Copying a sparse file through an interface that does not preserve holes can materialize zeros and consume the full logical size.

### **The Virtual File System and Mount Namespace**

The **Virtual File System (VFS)** is the kernel abstraction that lets common system calls dispatch to ext4, XFS, Btrfs, tmpfs, procfs, NFS, FUSE and other implementations. It does not force those filesystems to share an on-disk format. It defines common in-memory objects and method tables.

| VFS object | Represents | Typical state |
|---|---|---|
| Superblock | One mounted filesystem instance | Filesystem type, root, mount options, operations, global state |
| Inode | One filesystem object | Type, metadata, mapping/address space, inode operations |
| Dentry | One name in one parent lookup context | Name, parent, inode pointer or negative result, cache state |
| File | One open file description | Offset, status flags, credentials/context and file operations |
| Address space | Cached pages/folios for an inode or related object | Clean, dirty and writeback pages indexed by file offset |
| Mount | Connection between a filesystem root and a namespace mount point | Parent mount, mount point dentry, mounted root and flags |

![A mount namespace selects the visible path tree while VFS objects dispatch operations to different filesystem implementations.](assets/vfs-mount-namespace.svg){fig-alt="Two processes with different mount namespaces resolve /work through different mounted filesystems; common VFS file, dentry, inode, superblock, and cache objects dispatch to ext4, tmpfs, procfs, NFS, FUSE, or overlay implementations." width="97%"}

*Figure: original explanatory diagram based on Linux [VFS documentation](https://docs.kernel.org/filesystems/vfs.html) and [pathname lookup](https://docs.kernel.org/filesystems/path-lookup.html).*

A **mount** attaches the root of one filesystem at a directory in an existing tree. When path lookup reaches that mount point, it crosses to the mounted root. The covered directory still exists in the lower filesystem but is hidden through that mount until the upper filesystem is unmounted.

A **mount namespace** gives a process a particular view of the mount table and root. Two processes can resolve the same textual pathname to different filesystems because one namespace has a different mount at that point. This mechanism is foundational for containers, but visibility is not by itself a complete security boundary; the next chapter combines namespaces with credentials, capabilities, cgroups, system-call filtering and other controls.

Path lookup must tolerate concurrent rename, mount and unmount. Linux associates a path with both a dentry and a mount, uses sequence validation and references to prevent use-after-free, and can restart a fast RCU walk as a slower reference walk when a case requires blocking or stronger stabilization. A cached dentry accelerates lookup but does not permanently pin every object or bypass permission/revalidation rules.

<details>
<summary><strong>Shell inspection: relate a pathname to mounts, inode identity, allocation, and open handles</strong></summary>

```bash
PATH_TO_CHECK=/path/to/file

# Resolve each path component and show permissions/types.
namei -l "$PATH_TO_CHECK"

# Show the filesystem and mount that own this path.
findmnt -T "$PATH_TO_CHECK"

# Inspect persistent identity and size/allocation metadata.
stat "$PATH_TO_CHECK"
du -h "$PATH_TO_CHECK"

# On supporting local filesystems, show logical-to-physical extent information.
# Physical addresses are diagnostic and can change after defragmentation/COW.
filefrag -v "$PATH_TO_CHECK"

# See this process's exact mount view. Fields are documented by proc(5).
cat /proc/self/mountinfo

# List open descriptors in one process; permissions may restrict inspection.
ls -l "/proc/$PID/fd"
```

</details>

These commands describe different identities. A pathname can change while an open descriptor remains stable. Inode numbers are unique only within a filesystem identity. Physical extents can change without the pathname or inode number changing. A mount namespace can make an object unreachable by one path while another process still has it mounted or open.

### **Caching, Writeback, and Persistence Semantics**

Most regular-file I/O is buffered through the **page cache**. A read first looks for the file-offset range in cached pages/folios; a miss initiates the storage path from the previous chapter. A buffered write usually copies bytes into cached memory, marks pages dirty, updates in-memory inode state, and returns before storage I/O completes.

This decoupling improves performance because the kernel can:

- combine many small writes into larger requests;
- reorder writeback for locality and device parallelism;
- avoid writing data that is overwritten or deleted while still dirty;
- serve future reads from memory; and
- delay physical allocation until a better extent decision is possible.

It also creates a durability gap and requires backpressure when dirty memory grows too large.

#### **Page Cache and Buffer Cache**

The page cache is indexed principally by a file object's identity and file-page offset. `read()`, `write()` and shared `mmap()` normally interact with the same cached content, which is why a write through one interface becomes visible through another without rereading the device. Private mappings have copy-on-write semantics and do not write private modifications back to the file.

The term **buffer cache** historically describes cached raw device blocks and, in modern Linux discussion, often refers more narrowly to metadata/block buffers associated with page-cache memory. It is safer to name what is cached: file data pages, directory blocks, inode metadata, allocation maps or raw block-device ranges. The implementation is more unified than older diagrams showing two wholly independent caches.

![A buffered write moves a page from dirty through writeback to clean state or a recorded error.](assets/page-cache-writeback-states.svg){fig-alt="A write copies userspace bytes into a dirty page-cache folio and may return; later background or explicit writeback maps and submits I/O, then marks the page clean on success or records an error for synchronization calls." width="97%"}

*Figure: original explanatory diagram based on the Linux VFS [`address_space` model](https://docs.kernel.org/filesystems/vfs.html), [`fsync(2)`](https://man7.org/linux/man-pages/man2/fsync.2.html), and [OSTEP file-system implementation](https://pages.cs.wisc.edu/~remzi/OSTEP/file-implementation.pdf).*

A typical state transition is:

1. **Clean cached:** memory matches the backing state accepted by the filesystem.
2. **Dirty:** memory contains newer file data or metadata not yet written back.
3. **Writeback:** I/O is in progress; the page remains cached and may require coordination with new writers.
4. **Clean:** successful completion makes the cached version no newer than acknowledged backing state.
5. **Error:** failure is recorded so `fsync()`, `close()` or later operations can report it according to the filesystem/kernel contract.

Writeback can be triggered by explicit synchronization, dirty-age/ratio thresholds, memory pressure, periodic transaction commit, unmount/freeze or synchronous I/O flags. If dirty production outruns sustainable device service, the kernel must throttle writers. Otherwise volatile memory becomes an unbounded queue and latency explodes before eventual out-of-memory or write failure.

`O_DIRECT` asks supported file systems to minimize ordinary cache effects and transfer between user buffers and storage with strict alignment/coherence constraints. It is not a universal bypass, not automatically faster, and not a durability guarantee. Linux documents that `O_DIRECT` alone does not replace `O_SYNC`/synchronization when stable completion matters.

#### **fsync, Ordering, and Durability**

A successful `fsync(fd)` requests that modified file data and associated metadata required by its contract reach permanent storage, including a device-cache flush when the stack supports it. `fdatasync(fd)` may omit metadata unrelated to retrieving the data, such as some timestamp changes, but must include metadata such as file size when needed to read the new bytes correctly.

Important distinctions are:

| Mechanism | Primary guarantee | What it does not automatically guarantee |
|---|---|---|
| Successful buffered `write()` | Kernel accepted some/all bytes into the file abstraction/cache | Power-loss durability |
| `fdatasync(filefd)` | File data and retrieval-essential metadata synchronized | Parent directory entry persistence; all optional metadata |
| `fsync(filefd)` | File data plus associated file metadata synchronized | New/renamed parent directory entry unless directory is also synchronized |
| `fsync(dirfd)` | Directory changes covered by that filesystem's contract are synchronized | File contents unless that file was also synchronized as required |
| `O_DSYNC` / `O_SYNC` | Each write follows data-integrity or fuller synchronized completion semantics | A general transaction spanning several files |
| `sync` / `syncfs` | Broad writeback request for system/filesystem | A precise application commit protocol by itself |

Linux [`fsync(2)`](https://man7.org/linux/man-pages/man2/fsync.2.html) explicitly notes that synchronizing a file does not necessarily synchronize the directory entry that names it. Creating or renaming a file changes the parent directory, so durable publication commonly requires a separate directory `fsync()`.

![A durable same-directory replacement writes and synchronizes a temporary inode before rename, then synchronizes the parent directory.](assets/durable-replace-protocol-animated.svg){fig-alt="An animated protocol opens a parent directory, creates a unique temporary file, writes it, calls fdatasync, atomically renames it over the target, and fsyncs the parent directory, with crash-state interpretations between steps." width="98%"}

*Figure: original explanatory animation based on Linux [`fsync(2)`](https://man7.org/linux/man-pages/man2/fsync.2.html), [`rename(2)`](https://man7.org/linux/man-pages/man2/rename.2.html), and [`open(2)`](https://man7.org/linux/man-pages/man2/open.2.html).*

The classic replacement protocol separates **content construction**, **atomic visibility** and **durable naming**:

1. open the target's parent directory;
2. create a unique temporary file in that same directory/filesystem;
3. write the complete new contents and set required metadata;
4. synchronize the temporary file and handle delayed errors;
5. rename the temporary name over the target for atomic visibility; and
6. synchronize the parent directory before externally acknowledging durability.

Readers opening the target see the old or the new inode, not a file gradually overwritten in place. Existing descriptors for the old inode remain valid. The protocol is still one-file publication, not a transaction across arbitrary files, and exact remote/non-POSIX filesystem guarantees must be checked separately.

<details>
<summary><strong>C example: crash-conscious same-directory replacement on Linux/POSIX-style local filesystems</strong></summary>

```c
#define _POSIX_C_SOURCE 200809L
#include <errno.h>
#include <fcntl.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <sys/stat.h>
#include <sys/types.h>
#include <unistd.h>

static int write_all(int fd, const char *data, size_t length) {
    size_t done = 0;
    while (done < length) {
        ssize_t n = write(fd, data + done, length - done);
        if (n > 0) {
            done += (size_t)n;           // Successful writes can be partial.
        } else if (n < 0 && errno == EINTR) {
            continue;
        } else {
            return -1;
        }
    }
    return 0;
}

static void fail_before_rename(int dirfd, int tempfd,
                               const char *temp_name, const char *what) {
    int saved = errno;
    if (tempfd >= 0) close(tempfd);
    if (dirfd >= 0 && temp_name != NULL) {
        unlinkat(dirfd, temp_name, 0);   // Best-effort removal of unpublished temp.
    }
    if (dirfd >= 0) close(dirfd);
    errno = saved;
    perror(what);
    exit(EXIT_FAILURE);
}

int main(int argc, char **argv) {
    if (argc != 4 || strchr(argv[2], '/') != NULL) {
        fprintf(stderr, "usage: %s DIRECTORY TARGET_BASENAME CONTENT\n", argv[0]);
        return EXIT_FAILURE;
    }

    int dirfd = open(argv[1], O_RDONLY | O_DIRECTORY | O_CLOEXEC);
    if (dirfd == -1) {
        perror("open directory");
        return EXIT_FAILURE;
    }

    char temp_name[128];
    int used = snprintf(temp_name, sizeof temp_name,
                        ".replace-%ld.tmp", (long)getpid());
    if (used < 0 || (size_t)used >= sizeof temp_name) {
        errno = ENAMETOOLONG;
        fail_before_rename(dirfd, -1, NULL, "temporary name");
    }

    int tempfd = openat(dirfd, temp_name,
                        O_WRONLY | O_CREAT | O_EXCL | O_CLOEXEC, 0644);
    if (tempfd == -1) fail_before_rename(dirfd, -1, NULL, "openat temp");

    if (write_all(tempfd, argv[3], strlen(argv[3])) == -1) {
        fail_before_rename(dirfd, tempfd, temp_name, "write temp");
    }

    // Persist bytes and metadata required to retrieve them, including file size.
    if (fdatasync(tempfd) == -1) {
        fail_before_rename(dirfd, tempfd, temp_name, "fdatasync temp");
    }

    if (close(tempfd) == -1) {
        fail_before_rename(dirfd, -1, temp_name, "close temp");
    }
    tempfd = -1;

    // Same-directory rename atomically changes runtime namespace visibility.
    if (renameat(dirfd, temp_name, dirfd, argv[2]) == -1) {
        fail_before_rename(dirfd, -1, temp_name, "renameat");
    }

    // The rename succeeded, so failure from here is an uncertain durability result;
    // deleting the new target would be an incorrect attempt to roll back.
    if (fsync(dirfd) == -1) {
        int saved = errno;
        close(dirfd);
        errno = saved;
        perror("fsync directory (rename visible, durability uncertain)");
        return EXIT_FAILURE;
    }

    if (close(dirfd) == -1) {
        perror("close directory");
        return EXIT_FAILURE;
    }
    return EXIT_SUCCESS;
}
```

Compile and run:

```bash
cc -std=c11 -Wall -Wextra -O2 durable_replace.c -o durable_replace
./durable_replace ./state config.json '{"generation":2}'
```

</details>

The code uses a predictable per-process temporary name only to keep the example readable; production code should use collision-resistant creation and clean abandoned temporary files safely. Preserving ownership, mode, extended attributes or timestamps requires setting and synchronizing those properties before publication. If a directory `fsync()` fails after rename, the new name is already visible at runtime, so pretending to roll back by deleting it can make the situation worse.

The storage path below the filesystem matters. Filesystem ordering eventually becomes block requests plus flush/FUA/barrier semantics. A device that falsely reports volatile data as stable defeats the upper-layer promise. Conversely, forcing every write immediately destroys batching and latency, so correct applications place synchronization at meaningful commit boundaries rather than after every byte.

### **The Crash-Consistency Problem**

**Crash consistency** asks what on-disk states can be observed when execution stops between the writes needed for one logical operation. The desired behavior is often described as moving from one consistent state $S_0$ to another $S_1$ without exposing an invalid mixture.

Appending one new block to a file might require at least:

- `D`: write the new data block;
- `B`: mark that physical block allocated; and
- `I`: update the inode's size and mapping to point at it.

These are separate writes. If no ordering or atomic-group mechanism constrains $n$ writes, up to $2^n$ persisted subsets are conceptually possible. Strict ordering reduces this to prefixes, but a crash after an intermediate prefix can still leave leaks or stale structures. Atomic sectors/blocks also do not make a multi-block operation atomic.

![An append has dependent data, allocation, and inode updates whose partial persistence can leak or expose blocks.](assets/crash-update-dependencies.svg){fig-alt="Data block D, allocation bitmap B, and inode I form an ordered dependency for append; example crash subsets show stale data exposure, double allocation, leaked space, and the complete new state." width="97%"}

*Figure: original explanatory diagram based on [OSTEP crash consistency, fsck and journaling](https://pages.cs.wisc.edu/~remzi/OSTEP/file-journaling.pdf) and [UW CS537 crash-recovery notes](https://pages.cs.wisc.edu/~bart/537/lecturenotes/s27.html).*

Dangerous states include:

| Persistent subset | Consequence |
|---|---|
| Inode pointer without allocation bit | Another file can receive the same block; two owners corrupt each other |
| Inode pointer before initialized data | File exposes stale data from the block's previous owner or undefined contents |
| Allocation bit without inode pointer | Space leak: block appears occupied but unreachable |
| Directory entry without initialized inode | Name resolves to invalid metadata |
| Freed block before removing old pointer | Live object points into space available for reuse |
| New data durable but size not durable | Bytes exist physically but are outside the recovered logical file |

Some intermediate states are structurally consistent but semantically old or wasteful. Writing `D` before publishing `B` and `I` may leave an unreachable initialized block after a crash; recovery can reclaim it. Publishing `I` first can expose uninitialized or multiply owned storage and is much more dangerous. This motivates explicit **write dependencies**, but ordering alone still needs recovery or rollback for incomplete operations.

#### **Failure Models and Update Dependencies**

A recovery design should state which failures it covers:

| Failure | Typical observation | Needed mechanisms |
|---|---|---|
| Application/process crash | Kernel and storage continue; incomplete high-level operation | Atomic API operations, temp+rename, application WAL/version records |
| Kernel panic/power loss | Volatile memory disappears; device may contain a prefix/reordered subset | Filesystem journal/COW, flush ordering, recovery replay/root selection |
| Torn atomic unit | Part of a metadata/data block contains old/new mixture | Atomic-write assumptions, checksums, duplicated metadata, repair |
| Lost/misdirected/stale write | Correct location does not contain acknowledged version | Checksums with identity/generation, redundancy and scrub |
| Whole-device failure | No blocks available from that device | Replication/RAID plus tested recovery; backups for independent loss |
| Software bug or malicious update | Protocol itself writes a valid but wrong state | Isolation, validation, snapshots, auditing and independent backups |

The main on-disk invariants in a Unix-like teaching file system include:

- every referenced block is allocated;
- an ordinary block has at most one owner unless sharing/reference counts explicitly allow more;
- every reachable directory entry targets a valid allocated inode;
- inode link counts agree with directory references according to the format's rules;
- file size and mapping do not expose invalid ranges;
- free-space summaries agree with detailed maps; and
- recovery records are either complete and committed or safely ignored.

Dependencies can be enforced by synchronous ordering, a write-ahead log, copy-on-write root commit, soft updates, shadow paging or other protocols. The design must propagate ordering through CPU caches, filesystem queues, the block layer and volatile device caches. A source-code order such as `write(A); write(B);` is not by itself a persistence order.

Crash consistency is also not the same as an application transaction. A journal may ensure that a renamed file never points at an unallocated inode while still recovering either the old or new version. Whether the application was allowed to acknowledge the new version depends on its `fsync()` protocol. Several files updated by separate calls can be individually consistent yet mutually represent an impossible business state.

### **Recovery Techniques**

Recovery mechanisms differ mainly in **what evidence they leave before the crash**. A checker has only the final collection of blocks and must infer a plausible consistent state. A journal keeps a bounded description or copy of pending updates and uses a commit record to distinguish complete transactions from incomplete ones. A copy-on-write filesystem preserves the old tree while constructing the new one, then publishes a new root only after its descendants are stable.

| Technique | Evidence available after reboot | Normal-write cost | Recovery work | Main limitation |
|---|---|---:|---:|---|
| File-system checking | Existing metadata and structural invariants | Little protocol overhead | Potentially scans a large fraction of the filesystem | Repair can be slow and semantically ambiguous |
| Metadata journaling | Committed log transactions plus home locations | Extra metadata writes and ordering barriers | Replay a bounded journal tail | User data may still reflect an older version |
| Full-data journaling | Logged data and metadata | High write amplification | Replay committed transactions | Duplicates most writes |
| Copy-on-write trees | Old root and fully written candidate subtrees | Path copying, allocation and reference tracking | Select a valid committed root; later reclaim old blocks | Fragmentation and space-management complexity |

All three protect **filesystem structure**, but none alone replaces backups. If software overwrites the wrong file, an administrator deletes data intentionally, or the whole device is lost, the filesystem may remain perfectly consistent while the desired information is gone.

#### **File-System Checking**

A file-system checker such as `fsck` treats the disk image as a graph plus allocation accounting. It reads trusted bootstrap metadata, walks reachable directories and inodes, records which blocks and objects are claimed, then reconciles those observations with free-space maps and stored counters. The checker is reconstructing invariants after the fact; it is not replaying the exact intent of the interrupted operation.

![A file-system checker scans metadata, rebuilds observed ownership, compares it with allocation maps, and applies conservative repairs.](assets/fsck-reconciliation.svg){fig-alt="File-system checker pipeline: validate superblock, scan inodes, traverse directories, build observed block ownership, compare bitmaps and link counts, then repair or quarantine ambiguous objects." width="96%"}

*Figure: original explanatory diagram based on [OSTEP crash consistency and recovery](https://pages.cs.wisc.edu/~remzi/OSTEP/file-journaling.pdf) and [UW CS537 crash-recovery notes](https://pages.cs.wisc.edu/~bart/537/lecturenotes/s27.html).*

A simplified checker proceeds as follows:

```text
CHECK-FILE-SYSTEM(image)
    validate and choose a usable superblock copy
    observed_blocks <- empty ownership map
    observed_links  <- zero for every inode

    for each allocated inode i:
        validate type, size, timestamps, and block mappings
        for each block b referenced by i:
            record that i claims b
            flag b if out of range or multiply claimed

    traverse directories beginning at the root:
        validate each name -> inode entry
        increment observed_links[target inode]
        detect malformed cycles and unreachable subtrees

    compare observations with inode/block allocation maps
    repair unambiguous mismatches
    quarantine or ask about ambiguous objects
    recompute summaries and write repaired metadata safely
```

The two scans answer different questions. Walking allocated inodes discovers block claims even for files no longer reachable by any directory name. Walking the namespace discovers references and reachability. Their cross-check reveals common inconsistencies:

- a block marked free but referenced by an inode must not be reallocated;
- a block marked allocated but referenced nowhere can usually be reclaimed;
- two unrelated inodes claiming one ordinary block require choosing an owner or copying data;
- a reachable directory entry pointing to a free inode is invalid;
- an allocated, internally valid inode with no reachable name may be attached under a recovery directory such as `lost+found`; and
- a stored link count can be replaced by the count observed from directory entries.

Some repairs are logically safe but semantically unknowable. A checker can prove that an orphaned inode contains a plausible file; it cannot infer the original human-readable name or which project directory the user intended. Duplicate block ownership is worse: both files may contain valuable fragments, and no local invariant reveals which content is authoritative. Production tools therefore distinguish automatic low-risk repairs from actions that require policy, operator confirmation or read-only diagnosis.

The classic cost is proportional to filesystem metadata or capacity rather than the tiny set of blocks modified just before the crash. On multi-terabyte volumes, a full offline scan can dominate reboot time. Journals, checksums, duplicated superblocks, orphan lists and online scrub mechanisms reduce routine recovery cost, but checkers remain valuable when a bug, latent media error or corrupted journal violates the normal crash model.

#### **Journaling and Write-Ahead Logging**

**Journaling** applies write-ahead logging (WAL) to filesystem updates: record enough information in a dedicated log **before** the corresponding home-location metadata is allowed to become durable. A transaction's commit record is the recovery decision point. If the commit is valid, replay is safe; if it is absent or torn, the incomplete transaction is ignored.

![A journal transaction writes a descriptor and log records, commits them, checkpoints home locations, and replays only committed transactions after a crash.](assets/journal-wal-recovery-animated.svg){fig-alt="Animated write-ahead logging sequence: descriptor and metadata copies enter a circular journal, a commit record makes the transaction recoverable, home blocks are checkpointed, and reboot replays only complete committed records." width="97%"}

*Figure: original animated diagram based on the [Linux ext4 journal documentation](https://docs.kernel.org/filesystems/ext4/journal.html), [OSTEP journaling](https://pages.cs.wisc.edu/~remzi/OSTEP/file-journaling.pdf) and [UW CS537 WAL notes](https://pages.cs.wisc.edu/~bart/537/lecturenotes/s27.html).*

For a redo-style metadata journal, the core protocol is:

```text
COMMIT-JOURNAL-TRANSACTION(T)
    append descriptor(T: which home blocks will change)
    append new copies of T's journaled blocks
    force descriptor and copies to stable storage
    append and force COMMIT(T)
    acknowledge T as journal-committed

    later, write T's blocks to their home locations       // checkpoint
    after checkpoint is durable, reclaim T's journal space

RECOVER(journal)
    scan records using sequence numbers and checksums
    for each complete transaction with a valid COMMIT:
        copy its logged blocks to their home locations     // redo
    ignore incomplete or invalid trailing transactions
```

Redo is designed to be **idempotent**: replaying the same committed image twice produces the same home block. Sequence numbers prevent stale circular-log records from being mistaken for current work, checksums detect torn or corrupted records, and revoke records can prevent an obsolete logged image from overwriting a block that was later freed and reused.

Journaling normally batches many system calls into one filesystem transaction. Batching amortizes descriptor, commit and flush costs, while the circular journal bounds recovery: reboot scans the active tail rather than every inode on the volume. The trade-off is that a committed metadata block is often written twice, once to the journal and once to its home location. If $M$ bytes of metadata are journaled, the lower-bound metadata traffic is approximately $2M$ before accounting for descriptors, commits, alignment and cleaner effects.

What enters the journal defines the guarantee:

| Mode | What is logged | Critical ordering | Consequence after recovery |
|---|---|---|---|
| Metadata journaling with ordered data | Metadata copies; new file data goes to home blocks first | Data must be stable before metadata that exposes it commits | Structure is consistent and newly exposed ranges do not contain stale old-owner bytes |
| Writeback data mode | Metadata copies; data may reach disk before or after metadata | Fewer data-before-metadata constraints | Structure recovers, but recent file contents can be stale or surprising |
| Full data journaling | File data and metadata | Entire transaction reaches journal before commit | Strongest simple journal guarantee, with high write traffic |

Linux ext4 commonly uses metadata journaling with `data=ordered`: data associated with a metadata transaction is sent to its final location before that metadata transaction commits. `data=journal` writes both through the journal, while `data=writeback` relaxes the ordering. These are filesystem policies, not substitutes for the application's own `fsync()` commit point.

The distinction between three events is essential:

1. A system call has returned, so the operation is visible to processes using the current kernel state.
2. A filesystem transaction has committed to its journal, so its journaled structures can be replayed after reboot.
3. The application's intended unit of work is durable, including every required file and directory update.

Those events need not coincide. A journal can preserve internal allocation invariants without promising that an unsynchronized user file contains its newest bytes. Likewise, two independently journaled files do not automatically form one application-level transaction. Databases therefore keep their own WAL: the database knows which pages jointly represent a committed transaction, while the filesystem knows how to keep its block and namespace structures recoverable.

#### **Copy-on-Write File Systems**

A copy-on-write (COW) filesystem never overwrites blocks that belong to the currently committed tree while preparing an update. It writes modified leaves to new locations, recursively writes new ancestors that point to them, and only then publishes a new root reference. The old root remains a complete fallback until the new root is durably selected.

![Copy-on-write constructs a new leaf and ancestor path while preserving the old tree, then atomically publishes a new root.](assets/cow-tree-atomic-root.svg){fig-alt="Copy-on-write B-tree update: unchanged subtrees are shared, modified leaf and ancestors are written to new blocks, and a final root or superblock switch commits the new tree while the old root remains recoverable." width="96%"}

*Figure: original explanatory diagram based on the [Btrfs design documentation](https://btrfs.readthedocs.io/en/stable/dev/dev-btrfs-design.html) and the shadow-paging treatment in [OSTEP crash consistency](https://pages.cs.wisc.edu/~remzi/OSTEP/file-journaling.pdf).*

Suppose root $R_0$ reaches internal node $A_0$ and leaf $L_0$. Updating one record follows this dependency order:

```text
COW-UPDATE(key, value)
    L1 <- allocate copy of L0 with (key, value) changed
    write_and_verify(L1)

    A1 <- allocate copy of A0 pointing to L1
    write_and_verify(A1)

    R1 <- allocate new root pointing to A1 and unchanged subtrees
    write_and_verify(R1)

    publish_new_root(R1)       // small, carefully protected commit record
```

A crash before the final publication leaves $R_0$ authoritative; unreachable new blocks can be reclaimed. A crash after a valid root publication selects $R_1$. Real formats protect root selection with duplicated superblocks, generations, checksums and ordering rules because writing the root pointer itself can fail.

COW naturally enables snapshots and reflinks. A snapshot adds another root that references existing immutable blocks; subsequent updates copy only changed paths. Sharing requires reference counts or back-reference structures so a physical extent is freed only after its last logical owner disappears. Checksums fit naturally because a parent can store the checksum of the child version it references.

The price is displaced rather than eliminated:

- repeatedly changing a small logical region can fragment both data and metadata;
- changing one leaf may rewrite every tree node on its root path, known as the **wandering-tree problem**;
- reference accounting, delayed allocation and transaction aborts make low-space behavior difficult;
- snapshots retain old extents, so deleting a file may free much less space than expected;
- background balance, scrub and garbage collection compete with foreground I/O; and
- COW preserves old versions on the same storage system, so it is not an independent backup.

Journaling and COW are not mutually exclusive implementation labels: a system can combine COW trees with log-like structures for fast synchronization or use a journal alongside otherwise in-place data. The useful question is which structures are overwritten, what record marks a transaction committed, and how recovery validates that record.

| Question | Journaling / in-place home blocks | Copy-on-write tree |
|---|---|---|
| How is old committed state preserved? | Redo information in a bounded journal | Old blocks remain reachable from old root |
| Commit point | Valid journal commit record | Valid publication of a new root/generation |
| Post-commit cleanup | Checkpoint log copies to home locations | Reclaim blocks no longer referenced by any root |
| Snapshot fit | Requires separate machinery | Natural through shared immutable extents |
| Common performance concern | Double metadata writes and flush latency | Fragmentation, path copying and reference updates |

### **Log-Structured and SSD-Aware File Systems**

A **log-structured filesystem (LFS)** treats the log as the primary long-term layout: data blocks, inode versions and mapping information are accumulated in memory and appended in large sequential **segments**. This differs from journaling, where a usually bounded side log protects updates that will later live in conventional home locations.

![A log-structured filesystem appends new versions to segments and later cleans low-live-data victim segments.](assets/lfs-segment-cleaning.svg){fig-alt="Log-structured filesystem flow: buffered data and metadata are appended to a new segment with summaries, an inode map locates latest versions, and a cleaner copies live blocks from victim segments before reclaiming them." width="97%"}

*Figure: original explanatory diagram based on [OSTEP Log-structured File Systems](https://pages.cs.wisc.edu/~remzi/OSTEP/file-lfs.pdf) and the [Linux F2FS design documentation](https://docs.kernel.org/filesystems/f2fs.html).*

An LFS update does not seek back to overwrite every old block. It appends new versions and updates an inode map or node-address table that locates the latest metadata. Segment summaries identify which logical object and version produced each physical block. Recovery begins from checkpoints and scans a bounded tail to discover later complete updates.

Append-only placement makes foreground writes large and sequential, but old versions become garbage. A **segment cleaner** must:

1. choose victim segments, preferably those with little live data or old cold data;
2. read their summaries and verify which blocks are still current;
3. copy live blocks into new segments;
4. update mappings so the copies become current; and
5. erase or release the old segments for reuse.

Cleaning economics depend strongly on utilization. If a victim segment has live fraction $u$, copying its live data frees only fraction $1-u$ of the segment. Ignoring metadata and reads, the copying work per unit of space reclaimed grows roughly as

$$
\text{copy cost per reclaimed byte} \propto \frac{u}{1-u}.
$$

At $u=0.2$, copying $0.2$ segment frees $0.8$ segment; at $u=0.9$, copying $0.9$ frees only $0.1$. This is why free-space reserve, age-aware victim selection and hot/cold separation are structural performance policies rather than optional tuning.

For storage systems, **write amplification** is often summarized as

$$
WA = \frac{\text{physical bytes written below a layer}}
          {\text{logical bytes requested above that layer}}.
$$

An LFS can reduce small random foreground writes while cleaning raises filesystem-level $WA$. On flash, a separate flash translation layer (FTL) performs page remapping and erase-block garbage collection, producing another amplification layer. Filesystem cleaning and FTL cleaning can interfere if each moves data without knowing the other's liveness and temperature.

SSD-aware designs therefore consider properties absent from a simple magnetic-disk model:

- pages can be programmed, but an erase normally operates on a larger erase block;
- overwrites are implemented out of place by the device controller;
- discard/TRIM can tell the device which logical ranges no longer contain live data;
- write endurance and internal parallel channels affect placement policy;
- separating hot, warm and cold data reduces repeated movement of long-lived blocks; and
- zoned devices may require sequential writes within explicitly managed zones.

F2FS, for example, follows a log-structured approach but introduces multiple active logs for different data temperatures and a Node Address Table (NAT) that maps node identifiers to current block addresses. NAT reduces the classic wandering-tree cascade because a moved node does not require every ancestor pointer to be rewritten immediately. Checkpoints, segment summaries and roll-forward recovery work together to locate a valid state.

| Layout | Normal placement | Space reclamation | Best-case strength | Recurring cost |
|---|---|---|---|---|
| In-place indexed/extents | Rewrite mapped home regions | Free blocks/extents directly | Stable locality for mature files | Small random updates and crash ordering |
| Journaled in-place | Journal first, then home regions | Reclaim checkpointed journal tail | Fast bounded recovery | Journal traffic plus home writes |
| Copy-on-write tree | New extents and new tree paths | Reference-count/tree-based reclaim | Snapshots, checksums, atomic root commits | Fragmentation and metadata path copying |
| Log structured | Append new object versions in segments | Clean segments and copy live records | Batched sequential foreground writes | Cleaning amplification and free-space sensitivity |

No layout is universally fastest. Workload lifetime, overwrite frequency, free-space headroom, device behavior, required snapshots, recovery targets and operational tools all matter. A benchmark that measures only a fresh empty filesystem misses the long-run steady state where fragmentation or cleaning dominates.

### **Tracing the Running Case Through the File System**

Return to the chapter's running operation: safely replace `state/config.json` with a new generation and report success only when the replacement can survive the promised crash model. The end-to-end path joins the abstractions developed throughout this chapter.

![End-to-end durable replacement flows from pathname lookup through descriptors, cache and allocation to journal or COW commit and device persistence.](assets/fs-running-case-end-to-end.svg){fig-alt="End-to-end filesystem trace for durable replacement: resolve parent directory, create temporary inode, write through page cache, map extents and persist file, atomically rename, persist directory, recover via journal replay or copy-on-write root selection." width="98%"}

*Figure: original chapter synthesis grounded in the [Linux VFS](https://docs.kernel.org/filesystems/vfs.html), [`open(2)`](https://man7.org/linux/man-pages/man2/open.2.html), [`fsync(2)`](https://man7.org/linux/man-pages/man2/fsync.2.html), [`rename(2)`](https://man7.org/linux/man-pages/man2/rename.2.html) and the recovery sources cited above.*

The operation can be traced as follows:

1. `open("state", O_DIRECTORY)` begins pathname lookup from the process root/current-directory context. The VFS crosses mount boundaries, consults dentries and obtains the parent directory inode.
2. `openat(..., O_CREAT | O_EXCL)` allocates a new inode and a temporary directory entry. The returned descriptor refers to a new open file description; later renaming the pathname does not invalidate it.
3. `write()` copies bytes into page-cache pages and marks them dirty. Allocation may be delayed, so no final physical extent is required at the instant `write()` returns.
4. Writeback allocates blocks/extents, transforms dirty pages into I/O, and records metadata dependencies. Checksums, encryption and mapping layers may add work before requests reach the device.
5. `fdatasync(tempfd)` waits for the file data and retrieval-essential metadata, such as size and mappings, according to the filesystem/device error model. At this point the temporary file's content is durable, but the final name has not changed.
6. `renameat()` atomically changes namespace visibility within the same filesystem: concurrent lookup sees either the old target or the new target, not a half-name. The replaced inode may remain alive while another process holds it open.
7. `fsync(dirfd)` asks the filesystem to persist the directory-entry change. This step closes the common gap where the new file bytes survive but the rename disappears after reboot.
8. The filesystem commits the relevant journal transaction or COW root and sends required write/flush ordering through the block layer. Only completed commits are selected during recovery.

The crash point determines the allowed visible version:

| Crash point | Structurally acceptable recovery | Application interpretation |
|---|---|---|
| Before temporary file sync | Old target remains; incomplete temporary file may be absent or removable | Operation did not commit |
| After temporary file sync, before rename | Old target remains and durable temporary file may remain | Safe to retry or clean temp by policy |
| After rename, before directory sync | New target was visible before crash, but recovered name may be old or new depending on guarantees | Do not report durable success yet |
| After directory sync completes successfully | New target and name should survive the covered crash model | Commit can be acknowledged |
| Device reports a write/flush error | State and durability can be uncertain; later calls may report deferred error | Stop claiming success and enter explicit error recovery |

This protocol provides **whole-file replacement**, not a transaction spanning arbitrary files. Updating `config.json` and `index.json` as one indivisible state requires an application-level design: a manifest or generation pointer written last, a database transaction/WAL, or a filesystem transaction API with precisely documented semantics. Correct persistence starts by defining the logical commit unit.

<details>
<summary><strong>Observe the path with Linux inspection tools</strong></summary>

```bash
# Identify the mount and filesystem that own the target pathname.
findmnt -T ./state/config.json

# Compare pathname metadata and inode identity before and after replacement.
stat ./state/config.json

# Show logical-to-physical extents when the filesystem/tool supports it.
filefrag -v ./state/config.json

# Trace only the namespace, write, synchronization, and close calls.
# -ff follows child processes; -ttT shows time and syscall duration.
strace -ff -ttT \
  -e trace=openat,write,fdatasync,renameat,fsync,close \
  ./durable_replace ./state config.json '{"generation":3}'
```

Read the trace as a contract, not merely a call list. Check that data synchronization precedes publication, that rename occurs in one parent directory, and that a successful directory `fsync()` follows publication. `filefrag` is observational: physical extents may move later because of COW, defragmentation or cleaning, and some filesystems intentionally hide or virtualize placement.

</details>

### **Comparison and Summary**

A filesystem combines several independent policy choices. “It uses inodes” says how objects are described; “it uses extents” says how logical ranges map to storage; “it is journaled” says how interrupted updates recover; “it is copy-on-write” says how new versions are published; and “it is log structured” says where normal new versions are placed. Treating these labels as mutually exclusive whole-system categories obscures real designs.

| Layer or question | Core abstraction | What it solves | What it does not guarantee alone |
|---|---|---|---|
| Namespace | Directories, dentries, hard/symbolic links | Human-readable paths and sharing | Durable content or authorization policy |
| Open state | Descriptor and open file description | Per-process handle, offset and status | Path still exists or data is persistent |
| Object metadata | Inode/vnode | Type, owner, size, timestamps and mappings | One fixed physical location forever |
| Placement | Blocks, indexed maps, extents | Logical offset to storage mapping | Crash-safe multi-block updates |
| VFS/mounts | Common object operations and namespace composition | Multiple filesystem implementations under one API | Identical semantics for every filesystem |
| Cache/writeback | Page cache, dirty tracking and batching | Low-latency access and efficient I/O | `write()` return equals durable storage |
| Application commit | `fdatasync`/`fsync`, atomic rename, protocol | Explicit durability boundary | Multi-file transaction unless designed |
| Recovery | fsck, journal/WAL or COW root | Restores a valid structural state | Correct user intent, backup or security |

The most useful misconceptions to remove are:

| Misconception | More accurate model |
|---|---|
| “A filename is the file.” | A name is a directory entry that resolves to an inode-like object; links and open handles can outlive one name. |
| “A file descriptor is an inode number.” | A descriptor indexes per-process state, which references a system-wide open file description and then a filesystem object. |
| “`write()` means the bytes are on disk.” | `write()` usually establishes kernel-visible dirty state; persistence needs the documented synchronization protocol. |
| “`rename()` makes replacement durable.” | Same-filesystem rename gives atomic namespace visibility; directory synchronization addresses persistence of that name change. |
| “Journaling saves the newest file contents.” | The journal's guarantee depends on what is logged and ordered; it primarily protects recoverable filesystem structure. |
| “COW is automatically a backup.” | Snapshots share the same administrative and failure domain unless replicated independently. |
| “An SSD removes filesystem allocation concerns.” | Flash changes the cost model and adds FTL garbage collection; locality, lifetime and write amplification still matter. |
| “Crash consistency equals application correctness.” | A structurally valid old-or-new filesystem state may still violate a multi-file application invariant. |

When evaluating a persistence design, ask these questions in order:

1. **Object and name:** Which inode-like object does the path denote, and can links, rename or mounts change that relationship?
2. **Visibility:** At what operation does another process observe the new state?
3. **Dirty state:** Which bytes and metadata may still exist only in volatile caches?
4. **Ordering:** Which writes must become stable before another write may commit?
5. **Commit evidence:** Is completion represented by a journal commit, a COW root, a generation record or an application WAL?
6. **Recovery choice:** After each crash point, which old/new states are valid and how are incomplete blocks detected?
7. **Error path:** What happens when allocation, writeback or synchronization reports `ENOSPC`, `EIO` or delayed failure?
8. **Failure scope:** Does the design cover process crashes, power loss, torn writes, latent corruption, device loss and operator mistakes, or only a stated subset?

The chapter's central chain is therefore:

$$
\text{pathname}
\rightarrow \text{dentry/inode}
\rightarrow \text{open file state}
\rightarrow \text{page cache}
\rightarrow \text{block or extent mapping}
\rightarrow \text{ordered recovery transaction}
\rightarrow \text{stable device state}.
$$

Each arrow is a contract boundary. Understanding where visibility, ordering, error reporting and durability change meaning is more important than memorizing one filesystem's block format. The next chapter adds **protection and security** to this storage model: who may traverse a directory, open an inode, change metadata or cross an isolation boundary, and how the kernel mediates those decisions without re-explaining the persistence machinery developed here.
